# Project Phase 2

This phase builds directly on your Phase 1 proposal.

You are not building predictive models yet. You are learning how to understand your dataset using structured exploration and simple visualization.

The goal of this phase is to:

- Original Research Question
- Description of datasets used
- Explanation of performed joins
- Final dataset shape

This is your introduction to visualization as a reasoning tool, not a design competition.

---

**Submission Requirements**

Submit the following:

1. Jupyter Notebook (.ipynb)
2. All dataset files used
3. A short README section at the top of your notebook including:
    - Original research question
    - Description of datasets used
    - Explanation of any joins performed
    - Final dataset shape (rows × columns)


**Part 1: Dataset Preparation and Joins**

You must combine at least two datasets using Pandas.

You are required to use:

pd.merge()

You must clearly explain:

- What column you joined on
- What type of join you used (inner, left, etc.)
- Number of rows before the join
- Number of rows after the join
- Why the row count changed or stayed the same

Your final working dataset must:

- Contain at least 1,000 rows
- Contain at least 6 meaningful columns
- Include at least one categorical variable
- Include at least one numerical variable

Demonstrate:

- df.shape
- df.info()
- df.describe()


**Part 2: Required Exploratory Questions**

You must answer at least four specific EDA questions using visualization.

Each question must:

- Be clearly stated
- Use Pandas operations
- Include one iplot() visualization
- Include a written interpretation of 3 to 5 full sentences

You may not submit screenshots without explanation.

---

**Required Visualization Types**

You must include the following:

**1. Category Counts (Bar Chart)**

Example structure:

- df[‘category’].value_counts().iplot(kind=‘bar’)

Question type example:

- Which category appears most frequently?

---

**2. Grouped Aggregation (Bar Chart)**

Example structure:

- df.groupby(‘group’)[‘numeric’].mean().iplot(kind=‘bar’)

Question type example:

- Which group has the highest average value?

---

**3. Distribution (Histogram)**

Example structure:

- df[‘numeric’].iplot(kind=‘hist’)

Question type example:

- Is the distribution symmetric or skewed?
- Are there potential outliers?

---

**4. Trend or Top-N Comparison**

If time or ordered data exists:

- df.groupby(‘year’)[‘numeric’].mean().iplot(kind=‘line’)

OR
- df.sort_values(‘numeric’, ascending=False).head(10).iplot(kind=‘bar’)

Question type example:

- How does the main variable change over time?
- What are the top 10 highest values?


**Part 3: Required Interpretation**

For each visualization, answer the following in full sentences:

1. What question are you asking?
2. What method did you use?
3. What does the visualization show?
4. What insight can you draw from it?

Minimum 3 sentences per visualization.  
Clarity and reasoning matter more than aesthetics.

---

**Optional - for future reports**

- Identifying rows lost during joins
- Using Boolean filtering or query()
- Identifying missing data patterns


**Grading Criteria**

Your Phase 2 submission will be evaluated on:

- Correct implementation of join (25 points)
- Dataset meets size and structure requirements (15 points)
- Quality and clarity of EDA questions (20 points)
- Correct use of Pandas operations (15 points)
- Appropriate use of required visualizations (15 points)
- Depth and clarity of written interpretation (10 points)

**Total: 100 points**

---

This phase is about learning how to think with data.  
Plot → Observe → Interpret.  
Strong exploratory analysis will make modeling significantly easier in later phases.

# Our Implementation

## 1. Load the datasets

In [43]:
# Download the required datasets
import os
import urllib.request

os.makedirs("dataset", exist_ok=True)

file_path_1 = "dataset/cdc_depression_data.csv"
file_path_2 = "dataset/OxCGRT_simplified_v1.csv"
url_path_1 = "https://data.cdc.gov/api/views/8pt5-q6wp/rows.csv?accessType=DOWNLOAD"
url_path_2 = "https://github.com/OxCGRT/covid-policy-dataset.git"

if not os.path.exists(file_path_1):
    print("Downloading CDC depression dataset from data.gov ...")
    urllib.request.urlretrieve(
        url_path_1,
        file_path_1
    )
else:
    print("Dataset already exists. Skipping download.")

if not os.path.exists(file_path_2):
    print("Downloading OxCGRT dataset from GitHub ...")
    # Clone the repository and move the file
    os.system("git clone " + url_path_2)
    os.system("mv covid-policy-dataset/data/OxCGRT_simplified_v1.csv dataset/")
    os.system("rm -rf covid-policy-dataset")
else:
    print("Dataset already exists. Skipping download.")


Cloning into 'covid-policy-dataset'...
Updating files: 100% (31/31), done.


In [44]:
# Load the datasets
import pandas as pd

depression_data = pd.read_csv(file_path_1)
oxcgrt_data = pd.read_csv(file_path_2)

# Display the column names of each dataset
print("CDC Depression Dataset Columns:")
print(depression_data.columns)
print("\nOxCGRT Dataset Columns:")
print(oxcgrt_data.columns)

CDC Depression Dataset Columns:
Index(['Indicator', 'Group', 'State', 'Subgroup', 'Phase', 'Time Period',
       'Time Period Label', 'Time Period Start Date', 'Time Period End Date',
       'Value', 'Low CI', 'High CI', 'Confidence Interval', 'Quartile Range'],
      dtype='str')

OxCGRT Dataset Columns:
Index(['CountryName', 'CountryCode', 'RegionName', 'RegionCode',
       'Jurisdiction', 'Date', 'C1M_combined_numeric', 'C1M_combined',
       'C2M_combined_numeric', 'C2M_combined', 'C3M_combined_numeric',
       'C3M_combined', 'C4M_combined_numeric', 'C4M_combined',
       'C5M_combined_numeric', 'C5M_combined', 'C6M_combined_numeric',
       'C6M_combined', 'C7M_combined_numeric', 'C7M_combined',
       'C8EV_combined_numeric', 'C8EV_combined', 'E1_combined_numeric',
       'E1_combined', 'E2_combined_numeric', 'E2_combined',
       'H1_combined_numeric', 'H1_combined', 'H2_combined_numeric',
       'H2_combined', 'H3_combined_numeric', 'H3_combined',
       'H6M_combined_numeric'

/tmp/ipykernel_8770/2240304981.py:5: DtypeWarning: Columns (0: RegionName, 1: RegionCode, 2: MajorityVaccinated, 3: PopulationVaccinated) have mixed types. Specify dtype option on import or set low_memory=False.
  oxcgrt_data = pd.read_csv(file_path_2)


## II - Filter our datasets

In [45]:
# Step 1: Filter out columns we don't want to include in this assignment
# Filter the simplified OxCGRT dataset for the United States
oxcgrt_us = oxcgrt_data[oxcgrt_data['CountryName'] == 'United States']

# Compare the size of the dataset before and after filtering
print("\nSize of the original OxCGRT dataset:", oxcgrt_data.shape)
print("Size of the filtered OxCGRT dataset (United States):", oxcgrt_us.shape)

# TODO
# Fill missing values in the 'RegionName' column with 'United States' (as they are country-level data)
oxcgrt_us.loc[oxcgrt_us['RegionName'].isna() | (oxcgrt_us['RegionName'] == ''), 'RegionName'] = 'United States'

# Remove some unnecessary columns that we won't be using for our analysis
columns_to_drop = ['CountryName', 'CountryCode', 'RegionCode', 'Jurisdiction']
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Our dataset contains two versions of the measurements: *_combined and *_combined_numeric.
# For this assignment we will only keep the *_combined_numeric columns, as they are easier to work with for analysis and visualization.
# We might want to include the *_combined columns in a future assignment when we do more detailed analysis
columns_to_drop = [col for col in oxcgrt_us.columns if col.endswith('_combined') and not col.endswith('_combined_numeric')]
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Save the filtered dataset to a new CSV file
oxcgrt_us.to_csv(file_path_2, index=False)


Size of the original OxCGRT dataset: (390909, 50)
Size of the filtered OxCGRT dataset (United States): (56992, 50)


In [55]:
import pandas as pd
import numpy as np

# --- Build unique_time_periods = [Start, End, State] as int YYYYMMDD ---
temp = depression_data[['Time Period Start Date', 'Time Period End Date', 'State']].drop_duplicates().copy()

temp['Time Period Start Date'] = pd.to_datetime(
    temp['Time Period Start Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

temp['Time Period End Date'] = pd.to_datetime(
    temp['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

unique_time_periods = temp.values  # rows: [start, end, state]

# --- Ensure OxCGRT date is int YYYYMMDD ---
oxcgrt_us['Date'] = oxcgrt_us['Date'].astype('int64')

# --- Assign group to each OxCGRT row (State + Date in range) ---
oxcgrt_us['group'] = np.nan

for i, (start, end, state) in enumerate(unique_time_periods):
    mask = (
        (oxcgrt_us['RegionName'] == state) &
        (oxcgrt_us['Date'] >= start) &
        (oxcgrt_us['Date'] <= end)
    )
    oxcgrt_us.loc[mask, 'group'] = i

oxcgrt_us['group'] = pd.to_numeric(oxcgrt_us['group'], errors='coerce')

# --- Attach Start/End/State metadata to each OxCGRT row via group ---
group_meta = pd.DataFrame(unique_time_periods, columns=['Start', 'End', 'State'])
group_meta['group'] = np.arange(len(group_meta), dtype='int64')

oxcgrt_us = oxcgrt_us.merge(group_meta, on='group', how='left')

# --- Aggregate OxCGRT by group ---
# median for numeric columns, but DO NOT aggregate Date or group
num_cols = oxcgrt_us.select_dtypes(include='number').columns.difference(['Date', 'group'])

oxcgrt_aggregated = (
    oxcgrt_us
    .dropna(subset=['group'])
    .groupby('group', as_index=False)
    .agg({
        'RegionName': 'first',
        'State': 'first',
        'Start': 'first',
        'End': 'first',
        'MajorityVaccinated': 'first',
        **{c: 'median' for c in num_cols}
    })
)

# --- Convert CDC dates in full depression_data to int YYYYMMDD (to match) ---
depression_data['Time Period Start Date'] = pd.to_datetime(
    depression_data['Time Period Start Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

depression_data['Time Period End Date'] = pd.to_datetime(
    depression_data['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

# --- Merge (LEFT join keeps all CDC rows) ---
merged = depression_data.merge(
    oxcgrt_aggregated,
    left_on=['State', 'Time Period Start Date', 'Time Period End Date'],
    right_on=['State', 'Start', 'End'],
    how='left'
)

# Optional cleanup: drop duplicate State/RegionName or Start/End columns if you want
# merged = merged.drop(columns=['RegionName'])  # if redundant
# merged = merged.drop(columns=['Start', 'End'])  # if redundant

merged.to_csv('merged_data.csv', index=False)

In [ ]:
# Final command
# merged = depression_data.merge(oxcgrt_aggregated,
#                                on=['State', 'Time Period Start Date', 'Time Period End Date'],
#                                how='left')